[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/07_position_encoding.ipynb)

# 07. Position encoding

절대 위치를 더하는 방식에서 Q/K를 회전시키는 RoPE와 2D/3D axial 확장까지 진행한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Learned positional embedding

token embedding에 위치 vector를 더한다.


In [ ]:
T, D = 4, 6
tokens = torch.zeros(1, T, D, device=device)
pos = nn.Embedding(T, D).to(device)
idx = torch.arange(T, device=device)

y = tokens + pos(idx)[None]
print(y)


In [ ]:
_ = profile_call("learned PE", lambda z: z + pos(idx)[None], tokens)


## 2. Sinusoidal encoding

고정된 sin/cos frequency를 사용한다.


In [ ]:
position = torch.arange(T, device=device)[:, None]
freq = torch.exp(torch.arange(0, D, 2, device=device) * (-math.log(10000.0) / D))

pe = torch.zeros(T, D, device=device)
pe[:, 0::2] = torch.sin(position * freq)
pe[:, 1::2] = torch.cos(position * freq)

print(pe)


In [ ]:
_ = profile_call("sinusoidal PE", lambda: pe + 0)


## 3. RoPE

2차원 쌍마다 위치에 따른 회전을 적용한다.


In [ ]:
q = torch.tensor(
    [[[1., 0., 1., 0.],
      [1., 0., 1., 0.],
      [1., 0., 1., 0.],
      [1., 0., 1., 0.]]],
    device=device,
)

def rope_1d(x):
    T = x.size(-2)
    D = x.size(-1)
    half = D // 2
    angles = torch.arange(T, device=x.device)[:, None] * torch.arange(1, half + 1, device=x.device)[None]
    c, s = angles.cos(), angles.sin()

    a = x[..., :half]
    b = x[..., half:]
    return torch.cat([a * c - b * s, a * s + b * c], dim=-1)

print(rope_1d(q))


In [ ]:
_ = profile_call("RoPE 1D", rope_1d, q)


## 4. Axial 2D / 3D idea

각 공간축에 별도 위치 index를 주어 회전 각도를 구성한다.


In [ ]:
coords_2d = torch.cartesian_prod(
    torch.arange(2, device=device),
    torch.arange(2, device=device),
)
coords_3d = torch.cartesian_prod(
    torch.arange(2, device=device),
    torch.arange(2, device=device),
    torch.arange(2, device=device),
)

print("2D coords:\n", coords_2d)
print("3D coords:\n", coords_3d)


In [ ]:
_ = profile_call("3D coordinate creation", lambda: coords_3d + 0)


## References and provenance

**[7.1] Sinusoidal positional encoding**
- 출처: Vaswani et al., Attention Is All You Need
- 이 노트북에서 가져온 부분: fixed absolute position

**[7.2] RoPE**
- 출처: Su et al., RoFormer
- 이 노트북에서 가져온 부분: Q/K rotation by position

**[7.3] 2D/3D axial RoPE**
- 출처: modern ViT/DiT/video/3D transformer implementations, including Krea-family reports
- 이 노트북에서 가져온 부분: spatial-axis-specific rotary coordinates
